In [1]:
from pathlib import Path
from torch.utils.data import DataLoader

from rsc_toolkit.datasets.toy_dataset import ToyDataset
from rsc_toolkit.datasets.icbhi.index_dataset import IndexDataset

# Point to the toy wav+txt files (adjust if yours differs)
project_root = Path(".").resolve().parent.parent
toy_root = project_root / "sample_data" / "toy_dataset" / "files"

# Build an index CSV (writes to ./downloaded_datasets by default)
out_dir = project_root / "downloaded_datasets"
processor = ToyDataset.make_processor(
    outputs_directory=str(out_dir),
    raw_data_directory=str(toy_root),
    force=True,
    sampling_rate=16_000
    )

dataset = ToyDataset(downloader=ToyDataset.make_downloader(outputs_directory=str(out_dir)), processor=processor)

dataset.download()
processor.build_index(sample_level="cycle")  # or "clip"
index_csv = processor.index_dir_path / "index_cycle.csv"  # or index_clip.csv

# Load samples from the index (this yields dicts with "audio", "label", "sample_id", etc.)
ds = ICBHIIndexDataset(
    index_csv=index_csv,
    data_root=toy_root,
    sampling_rate=processor.sampling_rate,
    sample_level="cycle",
    split="train",
    split_file=project_root / "sample_data/toy_dataset/official_split.txt",
    )

print("num samples:", len(ds))
first = ds[0]
print(first["sample_id"], first["audio"].shape, first["label"])

loader = DataLoader(ds, batch_size=1, shuffle=True, num_workers=4)
batch = next(iter(loader))
print(batch["audio"].shape, batch["label"])


num samples: 231
107_2b3_Al_mc_AKGC417L_cycle_0.887_2.601 torch.Size([1, 27424]) 1


/Users/miika/miniconda3/envs/rsc_toolkit/lib/python3.9/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/Users/miika/miniconda3/envs/rsc_toolkit/lib/python3.9/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcode

torch.Size([1, 1, 40768]) tensor([1])


In [ ]:
from torch.utils.data import DataLoader, Dataset
import pathlib

from rsc_toolkit.datasets.feature_cache import FeatureCache
from rsc_toolkit.processors.clap import CLAPProcessor
from rsc_toolkit.processors.mel import MelBankAudioProcessor
from rsc_toolkit.datasets.icbhi.index_dataset import IndexFeatureDataset

cache_dir = out_dir / "features_cache"

# processor.preprocess_features_from_index(index_csv=index_csv, sample_level="cycle", batch_size=1, num_workers=0, cache_dir=cache_dir)

feature_cache = FeatureCache(cache_dir=cache_dir)

# fp = MelBankAudioProcessor(sampling_rate=processor.sampling_rate, n_mels=64, feature_cache=feature_cache)
fp = CLAPProcessor(
    sampling_rate=processor.sampling_rate,
    model_name="laion/clap-htsat-unfused",
    feature_cache=feature_cache
)

feature_cache.configure(fp.get_processor_config(), processor_name=fp.__class__.__name__)

feat_ds = IndexFeatureDataset(ds, fp)
feat_loader = DataLoader(feat_ds, batch_size=2, shuffle=True, num_workers=2)


In [7]:
feature_cache.get_cache_stats()

{'cache_dir': '/Users/miika/Desktop/personal/RSC-Toolkit/downloaded_datasets/features_cache/features/ddc821e5f8e903ec',
 'pt_files': 80,
 'pt_total_bytes': 6994003,
 'tmp_files': 0,
 'tmp_total_bytes': 0}

In [6]:

import torch
from torch import optim
import lightning as L
import torch.nn as nn

from rsc_toolkit.models.clap import AudioCLAP
from rsc_toolkit.metrics.util import accuracy

audio_clap = AudioCLAP(
    n_cls=4,
)

loss_func = nn.CrossEntropyLoss()


class PLAdapter(L.LightningModule):
    def __init__(self, rsc_model, loss_func):
        super().__init__()
        self.rsc_model = rsc_model
        self.loss_func = loss_func

    def training_step(self, batch, batch_idx):
        audio_batch = batch[0].squeeze(1)
        labels = batch[1]
        logits, loss = self.rsc_model.training_forward(
            audio_batch=audio_batch,
            labels=labels,
            loss_func=self.loss_func,
        )
        self.log("train_loss", loss)
        return loss
    
    def validation_step(self, batch, batch_idx):
        audio_batch = batch[0].squeeze(1)
        labels = batch[1]
        logits, loss = self.rsc_model.training_forward(
            audio_batch=audio_batch,
            labels=labels,
            loss_func=self.loss_func,
        )
        results, batch_size = accuracy(output=logits, target=labels)
        self.log("val_loss", loss)
        self.log("accuracy", torch.tensor(results), batch_size=batch_size)

    def configure_optimizers(self):
        optimizer = optim.Adam(self.parameters(), lr=1e-3)
        return optimizer

pl_model = PLAdapter(rsc_model=audio_clap, loss_func=loss_func)

In [7]:
trainer = L.Trainer(
    max_epochs=1,
    accelerator="cpu",
    log_every_n_steps=10,
    val_check_interval=10,
    limit_val_batches=2,
    )
trainer.fit(
    model=pl_model,
    train_dataloaders=feat_loader,
    val_dataloaders=feat_loader,
)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/miika/miniconda3/envs/rsc_toolkit/lib/python3.9/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.

  | Name      | Type             | Params | Mode  | FLOPs
---------------------------------------------------------------
0 | rsc_model | AudioCLAP        | 28.2 M | train | 0    
1 | loss_func | CrossEntropyLoss | 0      | train | 0    
---------------------------------------------------------------
28.2 M    Trainable params
0         Non-trainable params
28.2 M    Total params
112.772   Total estimated model params size (MB)
4         Modules in train mode
259       Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/Users/miika/miniconda3/envs/rsc_toolkit/lib/python3.9/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:485: Your `val_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for val/test dataloaders.
/Users/miika/miniconda3/envs/rsc_toolkit/lib/python3.9/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:429: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.
/Users/miika/miniconda3/envs/rsc_toolkit/lib/python3.9/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDec

Sanity Checking DataLoader 0:  50%|█████     | 1/2 [00:00<00:00,  2.95it/s]

/Users/miika/miniconda3/envs/rsc_toolkit/lib/python3.9/site-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 2. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.


...

Epoch 0:   9%|▊         | 10/116 [00:04<00:50,  2.11it/s, v_num=23]

/Users/miika/miniconda3/envs/rsc_toolkit/lib/python3.9/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/Users/miika/miniconda3/envs/rsc_toolkit/lib/python3.9/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcode

Epoch 0:  17%|█▋        | 20/116 [00:25<02:00,  0.80it/s, v_num=23]

/Users/miika/miniconda3/envs/rsc_toolkit/lib/python3.9/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/Users/miika/miniconda3/envs/rsc_toolkit/lib/python3.9/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcode

RuntimeError: Connection refused

zsh:1: command not found: tensorboard


In [34]:
fp.feature_cache.get_cache_stats()

AttributeError: 'NoneType' object has no attribute 'get_cache_stats'

In [ ]:
import os
from torch import optim, nn, utils, Tensor
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
import lightning as L

from rsc_toolkit.models.bts import BTS

# define the LightningModule
class LitAutoEncoder(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = BTS()

    def training_step(self, batch, batch_idx):

        self.model.
        

        loss = nn.functional.mse_loss(x_hat, x)
        self.log("train_loss", loss)
        return loss

    def configure_optimizers(self):
        optimizer = optim.Adam(self.parameters(), lr=1e-3)
        return optimizer


# init the autoencoder
autoencoder = LitAutoEncoder(encoder, decoder)